# Generate Datasets for CNN

Alex Keyken 17/11/26

Script for generating datasets of cropped digit images in different fonts.
First system fonts are filtered to only contain distinct Hindu-Arabic digits 1-9. Next, digit image sets are rendered randomly enriched with grid edges, rotations, gaussian blurs and center offsets to mimick real pictures. Finally, the generated sets are split into training, validation and test data, such that some fonts are only encountered in test data.

Reviewed and optimized using Claude Code.

In [1]:
import random
import numpy as np
from pathlib import Path
from functools import lru_cache
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw, ImageFilter, ImageFont

# Retrieve and Filter System Fonts

In [2]:
IMG_SIZE = 28
SCALE = 4 # smooth edges are created by rendering at bigger scale, then downsampling
CANVAS = IMG_SIZE * SCALE

FONT_EXTENSIONS = {".ttf", ".otf", ".ttc"}
SKIP_FONTS = ("dings", "symbol", "emoji", "icon", "math", "braille") # Fonts with symbols and emojis
FONT_DIRS = ["/System/Library/Fonts","/Library/Fonts","~/Library/Fonts"]

In [3]:
def find_fonts(directories: list[str]) -> list[Path]:
    """Return all font files found in the given directories."""
    found = set()
    for directory in directories:
        path = Path(directory).expanduser()
        if not path.is_dir():
            continue
        for file in path.rglob("*"):
            name = file.name.lower() # to eliminate SKIP_FONTS case sensitivity
            if file.suffix.lower() in FONT_EXTENSIONS and not any(k in name for k in SKIP_FONTS):
                found.add(file)
    return sorted(found)

def contains_digits(font_path: Path) -> bool:
    """Returns True if the font draws distinct, non-empty images for characters 1-9."""
    try:
        font = ImageFont.truetype(str(font_path), 64)
        shapes = set()
        for digit in "123456789":
            img = Image.new("L", (96, 96), 0) # create a 8-bit greyscale canvas
            ImageDraw.Draw(img).text((10, 10), digit, fill=255, font=font) # fill canvas with white digit
            if img.getbbox() is None: # if no digit is drawn, no bounding box will be found
                return False
            shapes.add(img.tobytes())
        return len(shapes) == 9
    except (OSError, ValueError):
        return False

# load_font wrapper to save function results and avoid rerunning
@lru_cache(maxsize=2048)
def load_font(path: str, size: int) -> ImageFont.FreeTypeFont:
    return ImageFont.truetype(path, size)

In [ ]:
# OPTIONAL DEBUG STEP: save image sets for each font in images/font_samples for manual filtering

def save_font_sample(font_path: Path, out_dir: Path = Path("images/font_samples")) -> None:
    """For debugging: render digits 1-9 in a font and save them as one PNG."""
    out_dir.mkdir(parents=True, exist_ok=True)
    
    font = ImageFont.truetype(str(font_path), 64)
    images = []
    for digit in range(1, 10):
        img = Image.new("L", (96, 96), 0)
        ImageDraw.Draw(img).text((10, 10), str(digit), fill=255, font=font)
        images.append(img)

    fig, axes = plt.subplots(1, 9, figsize=(14, 2))
    for ax, img, digit in zip(axes, images, range(1, 10)):
        ax.imshow(img, cmap="gray", vmin=0, vmax=255)
        ax.set_title(str(digit))
        ax.axis("off")
    fig.suptitle(f"{font_path.name}")
    plt.savefig(out_dir/f"{font_path.stem}.png", dpi=150)
    plt.close(fig)

font_paths = find_fonts(FONT_DIRS)
print(f"Found {len(font_paths)} fonts. Begin processing...")
n_font, n_rejected = 1,0
percentile = len(font_paths)//10
for font in font_paths:
    if contains_digits(font):
        save_font_sample(font)
    else:
        n_rejected += 1
    if n_font%percentile == 0 or n_font == len(font_paths):
        print(f"Scanned {n_font} fonts. Rejected {n_rejected}/{len(font_paths)} fonts.")
    n_font += 1

In [4]:
PROBLEMATIC_FONTS = [
    "Bodoni Ornaments", # symbols
    "NotoNastaliq", # out of bounds
    "Diwan Kufi", # out of bounds
    "DecoTypeNastaleeqUrdu", # out of bounds
    "Zapfino", # out of bounds
]

font_paths = find_fonts(FONT_DIRS)
filtered_font_paths = []
for font in font_paths:
    if contains_digits(font) and font.name not in PROBLEMATIC_FONTS:
        filtered_font_paths.append(font)

print(f"Found {len(filtered_font_paths)} usable fonts.")

Found 319 usable fonts.


# Generate Training Images

In [5]:
FRAC_INVERTED_IMG = 0.15
FRAC_GRID_EDGE = 0.4
FRAC_GAUSSIAN_BLUR = 0.3
MAX_DIGIT_ROTATION = 10 # deg

def render_cell(font_path: Path, digit: int, rng: random.Random,
                np_rng: np.random.Generator) -> np.ndarray:
    """Render a distorted cell image of the digit. Blank for digit == 0."""
    
    # default is white background, black foreground
    if rng.random() > FRAC_INVERTED_IMG:
        bg, fg = rng.randint(190, 255), rng.randint(0, 70)
    else:
        bg, fg = rng.randint(0, 60), rng.randint(185, 255)

    img = Image.new("L", (CANVAS, CANVAS), bg)
    draw = ImageDraw.Draw(img)

    # chance to draw grid lines along the cell edges, mimicking bad cropping
    line_colour = min(255, max(0, fg + rng.randint(-10, 10))) # grid color is similar to text color
    for edge in ("left", "right", "top", "bottom"):
        if rng.random() < FRAC_GRID_EDGE:
            width = rng.randint(1, 2) * SCALE
            offset = rng.randint(0, 2) * SCALE
            boxes = {
                "left": (offset, 0, offset + width, CANVAS),
                "right": (CANVAS - offset - width, 0, CANVAS - offset, CANVAS),
                "top": (0, offset, CANVAS, offset + width),
                "bottom": (0, CANVAS - offset - width, CANVAS, CANVAS - offset),
            }
            draw.rectangle(boxes[edge], fill=line_colour)

    if digit: # leave blank if digit == 0
        text = str(digit)
        font = load_font(str(font_path), int(CANVAS * rng.uniform(0.55, 0.85)))

        left, top, right, bottom = font.getbbox(text)

        # add up to 6% offset from center of box
        jitter = CANVAS * 0.06 
        x = (CANVAS - (right - left))/2 - left + rng.uniform(-jitter, jitter)
        y = (CANVAS - (bottom - top))/2 - top + rng.uniform(-jitter, jitter)

        # add up to MAX_DIGIT_ROTATION deg rotation
        mask = Image.new("L", (CANVAS, CANVAS), 0) # rotate on a mask to avoid rotating background
        ImageDraw.Draw(mask).text((x, y), text, fill=255, font=font)
        mask = mask.rotate(rng.uniform(-MAX_DIGIT_ROTATION, MAX_DIGIT_ROTATION), resample=Image.Resampling.BICUBIC)
        img.paste(fg, mask=mask)

    img = img.resize((IMG_SIZE, IMG_SIZE), Image.Resampling.LANCZOS) # resize scaled image

    # add gaussian blur to mimick bad image focusing
    if rng.random() < FRAC_GAUSSIAN_BLUR:
        img = img.filter(ImageFilter.GaussianBlur(rng.uniform(0.3, 1.0)))

    # after converting img to float, add pixel noise to mimick grainy images
    pixels = np.asarray(img, dtype=np.float32)
    pixels += np_rng.normal(0, rng.uniform(0, 10), pixels.shape)
    
    return np.clip(pixels, 0, 255).astype(np.uint8)

In [ ]:
# OPTIONAL DEBUG STEP: generate and save samples of distorted cell images

RNG_SEED = 123

rng = random.Random(RNG_SEED)
np_rng = np.random.default_rng(RNG_SEED)

def show_cell_variations(font_path: Path, digit: int, rng: random.Random,
                         np_rng: np.random.Generator, rows: int = 3, cols: int = 8,
                         out_dir: Path = Path("images/distortion_samples")) -> None:
    """For debugging: save digit sets to see the distortions."""
    out_dir.mkdir(parents=True, exist_ok=True)
    
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 1.2, rows * 1.3))
    for ax in axes.flat:
        img = render_cell(font_path, digit, rng, np_rng)
        ax.imshow(img, cmap="gray", vmin=0, vmax=255, interpolation="nearest")
        ax.axis("off")
    fig.suptitle(f"{font_path.name}, digit {digit}")
    plt.tight_layout()
    plt.savefig(out_dir/f"{digit}_{font_path.stem}.png", dpi=150)
    plt.close(fig)

for digit in range(0,10):
    font_index = rng.randint(0, len(filtered_font_paths)-1)
    font = filtered_font_paths[font_index]
    show_cell_variations(font, digit, rng, np_rng)

# Generate Training and Validation Datasets

In [6]:
IMG_PER_FONT = 10
RNG_SEED = 123

rng = random.Random(RNG_SEED)
np_rng = np.random.default_rng(RNG_SEED)

def build_dataset(fonts: list[Path], img_per_font: int, rng: random.Random,
                np_rng: np.random.Generator) -> tuple[np.ndarray, np.ndarray]:
    """
    Render images of every digit (0-9) for every font (img_per_font times),
    label and shuffle them.
    """
    images, labels = [], []
    for font in fonts:
        for digit in range(10):
            for _ in range(img_per_font):
                images.append(render_cell(font, digit, rng, np_rng))
                labels.append(digit)
                
    x = np.stack(images)
    y = np.array(labels, dtype=np.int64)
    order = np_rng.permutation(len(y))
    
    return x[order], y[order]

def save_preview(x: np.ndarray, y: np.ndarray, out_dir: Path = Path("images"), img_per_digit: int = 12) -> None:
    """For debugging: save a grid with one row per class, for a visual check."""
    out_dir.mkdir(parents=True, exist_ok=True)
    img_per_digit = min(img_per_digit, int((y == 0).sum()))
    
    rows = [np.concatenate(list(x[y == c][:img_per_digit]), axis=1) for c in range(10)]
    grid = Image.fromarray(np.concatenate(rows, axis=0))
    grid.resize((grid.width * 3, grid.height * 3), Image.Resampling.NEAREST).save(out_dir/"dataset_sample.png")

In [7]:
FRAC_TRAIN = 0.8
DATASET_FILE = f"datasets/{RNG_SEED}_sample.npz"

# randomize font order to reduce likelihood of similar fonts in the same dataset
rng.shuffle(filtered_font_paths)

# split fonts into training and validation sets
n_train_fonts = min(len(filtered_font_paths) - 1, max(1, round(len(filtered_font_paths)*FRAC_TRAIN)))
train_fonts, val_fonts  = filtered_font_paths[:n_train_fonts], filtered_font_paths[n_train_fonts:]
print(f"Training fonts: {len(train_fonts)}, validation fonts: {len(val_fonts)}")

x_train, y_train = build_dataset(train_fonts, IMG_PER_FONT, rng, np_rng)
x_val, y_val = build_dataset(val_fonts, IMG_PER_FONT, rng, np_rng)

np.savez_compressed(DATASET_FILE, x_train=x_train, y_train=y_train, x_val=x_val, y_val=y_val)
print(f"Saved {DATASET_FILE}: train {x_train.shape}, validation {x_val.shape}")

Training fonts: 255, validation fonts: 64
Saved datasets/123_sample.npz: train (25500, 28, 28), validation (6400, 28, 28)
